# Synchronization Primitives — Experiment Analysis

In [19]:
import subprocess
from pathlib import Path

import pandas as pd
import plotly.express as px

PYTHON_VERSIONS = ["3.13", "3.13t", "3.14", "3.14t"]

def run_benchmark(experiment: str, pythons: list = PYTHON_VERSIONS) -> pd.DataFrame:
    subprocess.run(
        ["nox", "-s", "experiments", "-p", *pythons, "--", experiment],
        check=True,
    )

    frames = []
    for python in pythons:
        path = Path(f"results/{experiment}_{python}.json")
        df = pd.read_json(path)
        df["python"] = python
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

In [22]:
df01 = run_benchmark("race_condition", pythons=["3.13"])
df01

nox > Running session experiments-3.13
nox > Creating virtual environment (uv) using python3.13 in .nox/experiments-3-13
nox > uv pip install pandas plotly
nox > python experiments/race_condition.py --output results/race_condition_3.13.json
nox > Session experiments-3.13 was successful in 35 seconds.


,python_version,case,tasks,result,expected,time_s,python
0,3.13.3,threading / unsafe,3,500,1500,0.626,3.13
1,3.13.3,threading / unsafe,5,500,2500,0.629,3.13
2,3.13.3,threading / unsafe,2,500,1000,0.633,3.13
3,3.13.3,threading / unsafe,4,500,2000,0.640,3.13
4,3.13.3,processes / unsafe,4,501,2000,0.932,3.13
5,3.13.3,processes / unsafe,2,501,1000,0.938,3.13
6,3.13.3,processes / unsafe,3,500,1500,0.944,3.13
7,3.13.3,processes / unsafe,5,500,2500,0.988,3.13
8,3.13.3,threading / safe,2,1000,1000,1.293,3.13
9,3.13.3,processes / safe,2,1000,1000,1.590,3.13


## 01. Race Condition

In [23]:
CASE_COLORS = {
    "threading / unsafe": "#93C4F9",  # light blue
    "threading / safe  ": "#1A6BC1",  # dark blue
    "processes / unsafe": "#F4B07A",  # light orange
    "processes / safe  ": "#C45A00",  # dark orange
}

px.bar(
    df01,
    x="tasks",
    y="time_s",
    color="case",
    color_discrete_map=CASE_COLORS,
    barmode="group",
    title="Execution time by concurrency level",
    labels={"time_s": "time (s)", "tasks": "concurrent tasks"},
)

## 02. Deadlock

In [ ]:
from experiments.deadlock import demo_fixed, demo_deadlock

## 03. strace — uncontended vs contended

In [ ]:
from experiments.strace_lock import uncontended, contended

## 04. GIL Benchmark

In [ ]:
from experiments.gil_benchmark import run_single, run_threaded

## 05. Lock Levels

In [ ]:
from experiments.lock_levels import bench_threading_lock, bench_asyncio_lock, bench_multiprocessing_lock